# ECON62020 - Group Replication Project

## Group
Guy Pigott - 8340277
Trinity Rose
Siqi Ge

## Replication Paper

Leora Friedberg, Elliott Isaac, "Same-Sex Marriage Recognition and Taxes: New Evidence about the Impact of Household Taxation", The Review of Economics and Statistics (2024) 106 (1): 85–101.

## Summary

This project replicates key results from Friedberg and Isaac (2024), which studies how changes in marriage-related tax incentives affect the likelihood of same-sex couples marrying. The central concept is the “marriage subsidy” (or penalty), defined as the difference between the taxes a couple would pay filing jointly versus as two single individuals.

The paper exploits variation in the timing of same-sex marriage recognition across U.S. states, as well as the 2013 federal recognition following United States v. Windsor, to identify the causal impact of tax incentives on marriage behaviour. To isolate exogenous variation, the authors construct a predicted marriage subsidy using LASSO-predicted earnings and use this as an instrument for the observed subsidy.

This replication focuses on reconstructing Table A2 of summary statistics, and reproducing the main results in Table 3, which shows the main OLS and IV regression results with and without controlling for income. The results confirm that higher marriage subsidies increase the probability of being married. While there are some differences in levels due to approximations in the tax simulation, the direction and relative magnitude of the effects are consistent with the original findings.

## Data Description

The analysis uses data derived from the American Community Survey (ACS) for the period 2003–2017. The unit of observation is a same-sex couple, identified from household-level data and classified as either married or cohabiting.

The dataset includes detailed demographic and income information for both partners. Income variables include wages, business income, investment income, retirement income, Social Security income, and government transfers. These are used to construct taxable income inputs for the NBER TAXSIM model.

The main outcome variable is an indicator for whether the couple is married. The key explanatory variable is the marriage subsidy, computed as the difference between joint tax liability and the sum of individual tax liabilities. Two versions are constructed: an observed subsidy based on reported income, and a predicted subsidy based on LASSO-predicted earnings.

Control variables include demographic characteristics (age, education, and differences between partners), household structure (number of children), and indicators for race and gender composition. The specification also includes state and year fixed effects, along with policy variables capturing same-sex marriage recognition and Medicaid expansion.

Due to data limitations, some tax inputs are approximated, which may lead to differences in levels relative to the original paper.

The code first scans a limited set of variables to identify households containing same-sex couples, using partner links (sploc), sex, detailed relationship codes, and a Census recode flag (qrelate). Households are classified as same-sex married or cohabiting, with an additional correction for couples recoded from spouse to unmarried partner.

A second pass reads a richer set of variables only for those households. The sample is restricted to adults aged 18–60 in U.S. states, excluding observations with allocated relationship values. Married and cohabiting couples are then assembled by matching the household head to the relevant partner, and duplicate households are removed.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

os.chdir("/Users/guypigott/python-venv-demo/EDS")
os.getcwd()

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Data" / "Output"

In [ ]:
#file_path = "/Users/guypigott/python-venv-demo/Replication Project/Data/Raw/acs_2012-2017.dat"

# Step 1: Read only 5 columns to find (year, serial) of same-sex couple households.
# These columns are sufficient to determine whether a person is in a same-sex couple.
#colspecs_scan = [
 #   (0,   4),    # year
  ##  (6,   14),   # serial
  #  (77,  81),   # pernum
   # (106, 108),  # sploc
   # (126, 130),  # related  (4-digit detailed version)
   # (130, 131),  # sex
   # (269, 270),  # qrelate
#]
#col_names_scan = ["year", "serial", "pernum", "sploc", "related", "sex", "qrelate"]

#print("Step 1: scanning for same-sex couples...")
#df_scan = pd.read_fwf(file_path, colspecs=colspecs_scan, names=col_names_scan)
#print(f"  Total rows: {len(df_scan):,}")

# ── Identify same-sex couples using vectorised operations ──────────────────────────
# Core logic: find partner via sploc, then compare sex values.

# Sort for consistent merging
#df_scan = df_scan.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Build a lookup table: (year, serial, pernum) → sex / related / qrelate
# Use merge instead of apply (much faster)
#partner_info = df_scan[["year","serial","pernum","sex","related","qrelate"]].copy()
#partner_info.columns = ["year","serial","sploc","partner_sex","partner_related","partner_qrelate"]

##df_scan = df_scan.merge(partner_info, on=["year","serial","sploc"], how="left")

# Same-sex flag
#same_sex    = df_scan["sex"] == df_scan["partner_sex"]
#sploc_valid = df_scan["sploc"] > 0

# Same-sex married: head (101) + spouse (201), or vice versa
#ss_mar = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 201)) |
  #  ((df_scan["related"] == 201)  & (df_scan["partner_related"] == 101))
#)

## Same-sex cohabiting: head (101) + unmarried partner (1114), or vice versa
# ss_coh = same_sex & sploc_valid & (
   # ((df_scan["related"] == 101)  & (df_scan["partner_related"] == 1114)) |
   # ((df_scan["related"] == 1114) & (df_scan["partner_related"] == 101))
#)

# mflag correction: qrelate==9 indicates the Census Bureau re-coded a same-sex
# spouse as "unmarried partner"; these should be treated as married.
#mflag = (df_scan["related"] == 1114) & (df_scan["qrelate"] == 9) & ss_coh

# Propagate mflag to the partner as well
#mflag_info = df_scan[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","partner_mflag"]
#df_scan = df_scan.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_scan["partner_mflag"] = df_scan["partner_mflag"].fillna(False)

#ss_mar = ss_mar | mflag | df_scan["partner_mflag"]
#ss_coh = ss_coh & ~mflag & ~df_scan["partner_mflag"]

#df_scan["ss_all"] = (ss_mar | ss_coh).astype(int)

# Collect the set of (year, serial) keys that belong to SS households
#ss_keys = df_scan.loc[df_scan["ss_all"] == 1, ["year","serial"]].drop_duplicates()
#print(f"  Same-sex couple households found: {len(ss_keys):,}")
#print(f"  Same-sex individuals: {df_scan['ss_all'].sum():,}")

# Core modification: Add the educd (160, 163) field directly to avoid reading the full file twice.
#colspecs_full = [
#    (0, 4), (6, 14), (37, 39), (77, 81), (106, 108), (118, 119), 
#    (124, 126), (126, 130), (130, 131), (131, 134), (134, 135), 
#    (147, 148), (151, 152), (158, 160), (160, 163), (193, 194), 
#    (251, 258), (261, 262), (269, 270)
#]
#col_names_full = [
 #   "year", "serial", "statefip", "pernum", "sploc", "nchild", 
#    "relate", "related", "sex", "age", "marst", 
#    "race", "hispan", "educ", "educd", "uhrswork", 
#    "incearn", "migrate1", "qrelate"
#]

# Create a hash set to speed up lookups
#ss_set = set(zip(ss_keys["year"], ss_keys["serial"]))
#chunks = []
#CHUNK = 500_000

#print("Step 2: reading full columns for SS households only...")
#reader = pd.read_fwf(file_path, colspecs=colspecs_full, names=col_names_full, chunksize=CHUNK)

#for chunk in reader:
#    mask = [(y, s) in ss_set for y, s in zip(chunk["year"], chunk["serial"])]
#   filtered = chunk[mask]
#   if len(filtered) > 0:
#        chunks.append(filtered)

#df_full = pd.concat(chunks, ignore_index=True)


# Calculate the number of adults in the household (used for subsequent precise filtering of cohabiting couples).
#df_full["adult"] = ((df_full["age"] >= 18) | (df_full["sploc"] != 0)).astype(int)
#num_adults = df_full.groupby(["year","serial"])["adult"].sum().reset_index(name="num_adults")
#df_full = df_full.merge(num_adults, on=["year","serial"], how="left")

#print(f"Done. Full data for SS households: {len(df_full):,} rows")

# Sort data to ensure merge logic is accurate
#df_clean = df_full.sort_values(["year","serial","pernum"]).reset_index(drop=True)

# Extract potential partner info and merge it back to the main table
#p_info = df_clean[["year","serial","pernum","sex","related","qrelate","marst"]].copy()
#p_info.columns = ["year","serial","sploc","p_sex","p_related","p_qrelate","p_marst"]
#df_clean = df_clean.merge(p_info, on=["year","serial","sploc"], how="left")

# Basic conditions: Same sex and valid spouse location (sploc)
#same_sex = df_clean["sex"] == df_clean["p_sex"]
#sploc_valid = df_clean["sploc"] > 0

# Identify same-sex married (ss_mar) and same-sex cohabiting (ss_coh)
#ss_mar = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 201)) | ((df_clean["related"] == 201) & (df_clean["p_related"] == 101)))
#ss_coh = same_sex & sploc_valid & (((df_clean["related"] == 101) & (df_clean["p_related"] == 1114)) | ((df_clean["related"] == 1114) & (df_clean["p_related"] == 101)))

# Handle mflag (married couples identified by qrelate==9)
#mflag = (df_clean["related"] == 1114) & (df_clean["qrelate"] == 9) & ss_coh
#mflag_info = df_clean[["year","serial","pernum"]].copy()
#mflag_info["mflag"] = mflag.values
#mflag_info.columns = ["year","serial","sploc","p_mflag"]

#df_clean = df_clean.merge(mflag_info, on=["year","serial","sploc"], how="left")
#df_clean["p_mflag"] = df_clean["p_mflag"].fillna(False)


# Finalize marriage and cohabitation classification
#ss_mar_final = ss_mar | mflag | df_clean["p_mflag"]
#ss_coh_final = ss_coh & ~mflag & ~df_clean["p_mflag"]

#df_clean["sscouple_mar"] = ss_mar_final.astype(int)
#df_clean["sscouple_coh"] = ss_coh_final.astype(int)
#df_clean["ss_any"] = (ss_mar_final | ss_coh_final).astype(int)

# Base filters: Age 18-60, exclude territories, exclude cases with allocated relationships
#mask_base = (
#    (df_clean["ss_any"] == 1) & 
#    (df_clean["age"] >= 18) & (df_clean["age"] <= 60) & 
#    (df_clean["statefip"] <= 56) & 
#    (df_clean["qrelate"] != 4) & (df_clean.get("p_qrelate", 0) != 4)
#)
#df_valid = df_clean[mask_base].copy()

# ================= Assemble Samples =================
# Married logic: Head (101) and their spouse
# mc = df_valid[df_valid["sscouple_mar"] == 1].copy()
#head_m = mc[mc["related"] == 101]
#spouse_m = mc[mc["related"].isin([201, 1114])]
#spouse_m = spouse_m.sort_values("pernum").groupby(["year","serial"]).first().reset_index()
#df_mar = head_m.merge(spouse_m, on=["year","serial"], suffixes=("_h","_s"))
#df_mar["married"] = 1

# Cohabiting logic: To reach 21,136, we typically do not restrict num_adults
#cc = df_valid[df_valid["sscouple_coh"] == 1].copy()
#head_c = cc[cc["related"] == 101]
#partner_c = cc[cc["related"] == 1114]
#df_coh = head_c.merge(partner_c, on=["year","serial"], suffixes=("_h","_s"))
#df_coh["married"] = 0

# Ensure bidirectional sploc matching (excludes complex cohabitation structures)
#df_coh = df_coh[(df_coh["sploc_h"] == df_coh["pernum_s"]) & (df_coh["sploc_s"] == df_coh["pernum_h"])]

# Concatenate and drop duplicates
#df_final = pd.concat([df_mar, df_coh], axis=0, ignore_index=True)
#df_final = df_final.drop_duplicates(subset=["year", "serial"])

#print(f"=== Final Sample (Target Alignment) ===")
#print(f"Married:    {len(df_final[df_final['married']==1]):,} (Paper: 16,098)")
#print(f"Cohabiting: {len(df_final[df_final['married']==0]):,} (Paper: 21,136)")

The final preparation stage constructs the variables used in estimation: partner earnings, total couple earnings, earnings split, age and education measures, race matching, children, and employment structure. Education is converted from detailed ACS codes into years of schooling, and policy variables are added for state marriage recognition, Windsor/Obergefell timing, and Medicaid expansion. The cleaned couple-level dataset is then saved for the later LASSO, TAXSIM, and regression stages.

In [ ]:
# Vectorized conversion: educd (detailed education code) to years of education
def educd_to_yrs_vec(s):
    s = s.fillna(-1).astype(int)
    result = np.full(len(s), np.nan)
    result[s <= 12] = 0
    result[s.isin([14, 15, 16, 17])] = s[s.isin([14, 15, 16, 17])] - 13
    result[s.isin([22, 23])] = s[s.isin([22, 23])] - 17
    result[s.isin([25, 26])] = s[s.isin([25, 26])] - 18
    result[s == 30] = 9
    result[s == 40] = 10
    result[s == 50] = 11
    result[(s >= 61) & (s <= 65)] = 12
    result[s == 71] = 13
    result[s == 81] = 14
    result[s == 101] = 16
    result[s >= 114] = 18
    return result

df_f = df_final.copy()